In [ ]:
from pathlib import Path
import sys

from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/pratik0620/adaptive-federated-aggregation.git"
REPO_ROOT = Path("/content/adaptive-federated-aggregation")
if not REPO_ROOT.exists():
    !git clone {REPO_URL} {REPO_ROOT}

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SRC_DIR = REPO_ROOT / "src"
if not SRC_DIR.exists():
    raise FileNotFoundError(f"src directory not found at {SRC_DIR}")

print("Repository:", REPO_ROOT)
print("Source modules:", SRC_DIR)

In [ ]:
# Install the packages required by the validation notebook and shared modules.
%pip install -q numpy pandas scikit-learn matplotlib jupyter


# Validate federated metadata clients

This notebook validates each client independently and never pools them into a training dataframe. It checks class counts, metadata quality, duplicates, patient/study leakage, and produces comparison plots.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.preprocessing_utils import assert_no_leakage, assert_no_study_leakage

BASE_DRIVE_PATH = Path("/content/drive/MyDrive/Healthcare_FL")
CLIENT_DIRS = {
    'NIH': BASE_DRIVE_PATH / 'processed' / 'client_01_NIH',
    'CheXpert': BASE_DRIVE_PATH / 'processed' / 'client_02_CheXpert',
    'PadChest': BASE_DRIVE_PATH / 'processed' / 'client_03_PadChest',
}
CORE = ['image_id','patient_id','study_id','source','target','age','sex','view_position','projection','image_width','image_height','pixel_spacing_x','pixel_spacing_y','scanner_manufacturer','image_path']
clients = {}
print("Google Drive:", BASE_DRIVE_PATH)
for source, directory in CLIENT_DIRS.items():
    paths = {split: directory / f'{split}.csv' for split in ['train','val','test']}
    missing = [str(path) for path in paths.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError(f'{source} client is incomplete: {missing}')
    clients[source] = {split: pd.read_csv(path) for split, path in paths.items()}
    print(source, {split: len(frame) for split, frame in clients[source].items()})

In [ ]:
rows = []
for source, splits in clients.items():
    assert all(set(CORE).issubset(frame.columns) for frame in splits.values())
    assert_no_leakage(splits)
    assert_no_study_leakage(splits)
    full = pd.concat(splits.values(), ignore_index=True)
    duplicate_images = int(full['image_id'].duplicated().sum())
    duplicate_patients_within_client = int(full['patient_id'].duplicated().sum())
    positive = int((full['target'] == 1).sum())
    negative = int((full['target'] == 0).sum())
    rows.append({'Client': source, 'Source': full['source'].dropna().iloc[0] if full['source'].notna().any() else source, 'Images': len(full), 'Patients': full['patient_id'].nunique(), 'Studies': full['study_id'].nunique(), 'Positive': positive, 'Negative': negative, 'Positive %': round(100 * positive / len(full), 2) if len(full) else np.nan, 'Duplicate image IDs': duplicate_images, 'Duplicate patient rows': duplicate_patients_within_client})
    print(source, 'missing values')
    display(full.isna().mean().sort_values(ascending=False).to_frame('missing_fraction'))
    print('patient/study split checks: passed')
summary = pd.DataFrame(rows)
display(summary)
print('Cross-client patient IDs are reported, not treated as leakage: sources may use independent ID namespaces.')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for index, (source, splits) in enumerate(clients.items()):
    full = pd.concat(splits.values(), ignore_index=True)
    full['target'].value_counts().sort_index().rename(index={0:'negative',1:'positive'}).plot.bar(ax=axes[0,index], title=f'{source}: class')
    pd.Series({name: len(frame) for name, frame in splits.items()}).plot.bar(ax=axes[1,index], title=f'{source}: samples')
    axes[0,index].set_ylabel('images'); axes[1,index].set_ylabel('images')
plt.tight_layout(); plt.show()
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for index, (source, splits) in enumerate(clients.items()):
    full = pd.concat(splits.values(), ignore_index=True)
    pd.to_numeric(full['age'], errors='coerce').dropna().plot.hist(ax=axes[index], bins=20, title=f'{source}: age')
    axes[index].set_xlabel('age')
plt.tight_layout(); plt.show()
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for index, (source, splits) in enumerate(clients.items()):
    full = pd.concat(splits.values(), ignore_index=True)
    full.isna().mean().sort_values(ascending=False).head(10).plot.bar(ax=axes[index], title=f'{source}: missing metadata')
    axes[index].tick_params(axis='x', rotation=75)
plt.tight_layout(); plt.show()